[**Open In Colab**](https://colab.research.google.com/github/HassanAlgoz/agentic-ai-systems/blob/main/Lessons/L01/05_RAG_agent.ipynb)

# RAG Chain and RAG Agent

[Source: Build a RAG agent with LangChain | LangChain](https://docs.langchain.com/oss/python/langchain/rag).

## Overview

One of the most powerful applications enabled by LLMs is sophisticated question-answering (Q\&A) chatbots. These are applications that can answer questions about specific source information. These applications use a technique known as Retrieval Augmented Generation, or [RAG](https://docs.langchain.com/oss/python/langchain/retrieval/).

This tutorial will show how to build a simple Q\&A application over an unstructured text data source. We will demonstrate:

1. A **RAG chain** that uses just a single LLM call per query. This is a fast and effective method for simple queries.
2. A **RAG agent** that executes searches with a simple tool. This is a good general-purpose implementation.


### Concepts

We will cover the following concepts:

* **Indexing**: a pipeline for ingesting data from a source and indexing it. *This usually happens in a separate process.*

* **Retrieval and generation**: the actual RAG process, which takes the user query at run time and retrieves the relevant data from the index, then passes that to the model.

Once we've indexed our data, we will use an [agent](https://docs.langchain.com/oss/python/langchain/agents) as our orchestration framework to implement the retrieval and generation steps.

## Setup


### Installation

This tutorial requires these langchain dependencies:

```bash
%pip install langchain langchain-core "langchain[openai]" langchain-text-splitters langchain-community bs4 langchain-huggingface sentence-transformers
```

For more details, see our [Installation guide](https://docs.langchain.com/oss/python/langchain/install).


### LangSmith

Many of the applications you build with LangChain will contain multiple steps with multiple invocations of LLM calls. As these applications get more complex, it becomes crucial to be able to inspect what exactly is going on inside your chain or agent. The best way to do this is with [LangSmith](https://smith.langchain.com).

After you sign up at the link above, make sure to set your environment variables to start logging traces:

```shels
export LANGSMITH_TRACING="true"
export LANGSMITH_API_KEY="..."
```

Or, set them in Python:

```python
import getpass
import os

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = getpass.getpass()
```

## Components

We will need to select **three components** from LangChain's suite of integrations.

#### 1. Select a chat model

👉 Read the [OpenAI chat model integration docs](https://docs.langchain.com/oss/python/integrations/chat/openai/)

In [ ]:
import os
from google.colab import userdata

# We use OpenRouter for the agent — add OPENROUTER_API_KEY to Colab Secrets (key icon in left sidebar)
# Get your key at https://openrouter.ai/keys
os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")

In [5]:
from langchain_openai import ChatOpenAI

# https://openrouter.ai/nvidia/nemotron-3-nano-30b-a3b:free
model_nemotron3_nano = ChatOpenAI(
    model="nvidia/nemotron-3-nano-30b-a3b:free",
    temperature=0,
    # OpenRouter instead of the default OpenAI API
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY"),
)

/home/halgoz/work/ai-agents/content/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### 2. Select an embeddings model

We use HuggingFace with the `all-mpnet-base-v2` sentence transformer:

In [6]:
# from langchain_openai import OpenAIEmbeddings
# embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

#### 3. Select a vector store

In [7]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)


## 1. Indexing

![Index Diagram](https://mintcdn.com/langchain-5e9cc07a/I6RpA28iE233vhYX/images/rag_indexing.png?fit=max&auto=format&n=I6RpA28iE233vhYX&q=85&s=21403ce0d0c772da84dcc5b75cff4451)


### Loading documents

We need to first load the blog post contents. We can use [DocumentLoaders](https://docs.langchain.com/oss/python/langchain/retrieval#document_loaders) for this, which are objects that load in data from a source and return a list of [Document](https://reference.langchain.com/python/langchain_core/documents/#langchain_core.documents.base.Document) objects.

In this case we'll use the [`WebBaseLoader`](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base), which uses `urllib` to load HTML from web URLs and `BeautifulSoup` to parse it to text. We can customize the HTML -> text parsing by passing in parameters into the `BeautifulSoup` parser via `bs_kwargs` (see [BeautifulSoup docs](https://beautiful-soup-4.readthedocs.io/en/latest/#beautifulsoup)). In this case only HTML tags with class “post-content”, “post-title”, or “post-header” are relevant, so we'll remove all others.

In [8]:
import bs4
from langchain_community.document_loaders import WebBaseLoader

# Only keep post title, headers, and content from the full HTML.
bs4_strainer = bs4.SoupStrainer(class_=("post-title", "post-header", "post-content"))
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs={"parse_only": bs4_strainer},
)
docs = loader.load()

assert len(docs) == 1
print(f"Total characters: {len(docs[0].page_content)}")

USER_AGENT environment variable not set, consider setting it to identify your requests.


Total characters: 43047


In [9]:
print(docs[0].page_content[:500])



      LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.
Agent System Overview#
In



**Go deeper**

`DocumentLoader`: Object that loads data from a source as list of `Documents`.

* [Integrations](https://docs.langchain.com/oss/python/integrations/document_loaders/): 160+ integrations to choose from.
* [`BaseLoader`](https://reference.langchain.com/python/langchain_core/document_loaders/#langchain_core.document_loaders.BaseLoader): API reference for the base interface.


### Splitting documents

Our loaded document is over 42k characters which is too long to fit into the context window of many models. Even for those models that could fit the full post in their context window, models can struggle to find information in very long inputs.

To handle this we'll split the [`Document`](https://reference.langchain.com/python/langchain_core/documents/#langchain_core.documents.base.Document) into chunks for embedding and vector storage. This should help us retrieve only the most relevant parts of the blog post at run time.

As in the [semantic search tutorial](https://docs.langchain.com/oss/python/langchain/knowledge-base), we use a `RecursiveCharacterTextSplitter`, which will recursively split the document using common separators like new lines until each chunk is the appropriate size. This is the recommended text splitter for generic text use cases.

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # chunk size (characters)
    chunk_overlap=200,  # chunk overlap (characters)
    add_start_index=True,  # track index in original document
)
all_splits = text_splitter.split_documents(docs)

print(f"Split blog post into {len(all_splits)} sub-documents.")

Split blog post into 63 sub-documents.




**Go deeper**

`TextSplitter`: Object that splits a list of [`Document`](https://reference.langchain.com/python/langchain_core/documents/#langchain_core.documents.base.Document) objects into smaller
chunks for storage and retrieval.

* [Integrations](https://docs.langchain.com/oss/python/integrations/splitters/)
* [Interface](https://python.langchain.com/api_reference/text_splitters/base/langchain_text_splitters.base.TextSplitter.html): API reference for the base interface.


### Storing documents

Now we need to index our 66 text chunks so that we can search over them at runtime. Following the [semantic search tutorial](https://docs.langchain.com/oss/python/langchain/knowledge-base), our approach is to [embed](https://docs.langchain.com/oss/python/langchain/retrieval#embedding_models/) the contents of each document split and insert these embeddings into a [vector store](https://docs.langchain.com/oss/python/langchain/retrieval#vectorstores/). Given an input query, we can then use vector search to retrieve relevant documents.

We can embed and store all of our document splits in a single command using the vector store and embeddings model selected at the [start of the tutorial](https://docs.langchain.com/oss/python/langchain/rag#components).

In [11]:
document_ids = vector_store.add_documents(documents=all_splits)

print(document_ids[:3])

['bf22b9d9-98a8-4d96-805a-16ba32726bf6', '12d6b8e4-cac4-4171-b8c3-670300580a45', '96efbfe6-8e4e-4e6d-8af6-e9ef8021a70f']


**Go deeper**

`Embeddings`: Wrapper around a text embedding model, used for converting text to embeddings.

* [Integrations](https://docs.langchain.com/oss/python/integrations/text_embedding/): 30+ integrations to choose from.
* [Interface](https://reference.langchain.com/python/langchain_core/embeddings/#langchain_core.embeddings.embeddings.Embeddings): API reference for the base interface.

`VectorStore`: Wrapper around a vector database, used for storing and querying embeddings.

* [Integrations](https://docs.langchain.com/oss/python/integrations/vectorstores/): 40+ integrations to choose from.
* [Interface](https://python.langchain.com/api_reference/core/vectorstores/langchain_core.vectorstores.base.VectorStore.html): API reference for the base interface.

This completes the **Indexing** portion of the pipeline. At this point we have a query-able vector store containing the chunked contents of our blog post. Given a user question, we should ideally be able to return the snippets of the blog post that answer the question.


## 2. Retrieval and generation

RAG applications commonly work as follows:

1. **Retrieve**: Given a user input, relevant splits are retrieved from storage using a [Retriever](https://docs.langchain.com/oss/python/langchain/retrieval#retrievers).
2. **Generate**: A [model](https://docs.langchain.com/oss/python/langchain/models) produces an answer using a prompt that includes both the question with the retrieved data

![Retrieval Diagram](https://mintcdn.com/langchain-5e9cc07a/I6RpA28iE233vhYX/images/rag_retrieval_generation.png?fit=max&auto=format&n=I6RpA28iE233vhYX&q=85&s=994c3585cece93c80873d369960afd44)


Now let's write the actual application logic. We want to create a simple application that takes a user question, searches for documents relevant to that question, passes the retrieved documents and initial question to a model, and returns an answer.

We will demonstrate:

1. A two-step RAG [chain](#rag-chains) that uses just a single LLM call per query. This is a fast and effective method for simple queries.
2. A RAG [agent](#rag-agents) that executes searches with a simple tool. This is a good general-purpose implementation.


### 1. RAG chains

In this approach we no longer call the model in a loop, but instead make a single pass, in a **two-step chain**:

1. run a search (potentially using the raw user query) and
2. incorporate the result as context for a single LLM query.

This results in a single inference call per query, buying reduced latency at the expense of flexibility.


We can implement this chain by removing tools from the agent and instead incorporating the retrieval step into a custom prompt:

In [ ]:
from langchain.messages import HumanMessage, SystemMessage

def retrieve_context(query: str):
    """Retrieve information to help answer a query."""
    retrieved_docs = vector_store.similarity_search(query, k=2)
    serialized = "\n\n".join(
        (f"Source: {doc.metadata}\n"
        f"Content: {doc.page_content}")
        for doc in retrieved_docs
    )
    return serialized


def rag_chain(query: str):
    """Inject context into prompt and return the LLM response."""
    prompt = (
        "You are a helpful assistant. Use the following context in your response:"
        "\n\n{context}"
    )
    context = retrieve_context(query)
    return model_nemotron3_nano.invoke([
        SystemMessage(prompt.format(context=context)),
        HumanMessage(query)
    ])


In [ ]:
ai_message = rag_chain("What is task decomposition?")

In [ ]:
ai_message.pretty_print()

================================== Ai Message ==================================

**Task decomposition** is the process of breaking a complex, high‑level goal into a series of smaller, more manageable sub‑tasks or steps that can be tackled individually.  

In the context you provided, task decomposition can be achieved in three main ways:

1. **LLM‑driven prompting** – asking the language model to generate the steps itself, e.g., “Steps for XYZ \n1.” or “What are the subgoals for achieving XYZ?”  
2. **Task‑specific instructions** – giving the model a prompt that implicitly defines the decomposition, such as “Write a story outline” for novel writing.  
3. **Human input** – having a person explicitly outline the sub‑tasks that need to be completed.

These approaches turn a single, difficult problem into a sequence of simpler problems, making planning and execution easier for the agent. (The context also links this idea to techniques like Chain‑of‑Thought and Tree‑of‑Thought, which furth

In the [LangSmith trace](https://smith.langchain.com/public/0322904b-bc4c-4433-a568-54c6b31bbef4/r/9ef1c23e-380e-46bf-94b3-d8bb33df440c) we can see the retrieved context incorporated into the model prompt.

This is a fast and effective method for simple queries in constrained settings, when we typically do want to run user queries through semantic search to pull additional context.

### 2. RAG agents

One formulation of a RAG application is as a simple [agent](https://docs.langchain.com/oss/python/langchain/agents) with a tool that retrieves information. We can assemble a minimal RAG agent by implementing a [tool](https://docs.langchain.com/oss/python/langchain/tools) that wraps our vector store:

In [ ]:
from typing import Literal
from langchain.tools import tool

@tool(response_format="content_and_artifact")
def retrieve_context_2(
    query: str,
    # section can help the Agent decide where it
    # wants to focus its attention
    # no need to actually use the value
    section: Literal["beginning", "middle", "end"]
):
    """Retrieve information to help answer a query."""
    retrieved_docs = vector_store.similarity_search(query, k=2)
    serialized = "\n\n".join(
        (f"Source: {doc.metadata}\n"
        f"Content: {doc.page_content}")
        for doc in retrieved_docs
    )
    content, artifact = serialized, retrieved_docs
    return content, artifact


::: {.callout-tip}
Here we use the [tool decorator](https://reference.langchain.com/python/langchain/tools/#langchain.tools.tool) to configure the tool to attach raw documents as [artifacts](https://docs.langchain.com/oss/python/langchain/messages#param-artifact) to each [ToolMessage](https://docs.langchain.com/oss/python/langchain/messages#tool-message). This will let us access document metadata in our application, separate from the stringified representation that is sent to the model.
:::


Given our tool, we can construct the agent:

In [23]:
from langchain.agents import create_agent


agent = create_agent(
    model=model_nemotron3_nano,
    tools=[retrieve_context],
    # System prompt is optional
    system_prompt=(
        "You have access to a tool that retrieves context from a blog post. "
        "Use the tool to help answer user queries."
    )
)


Let's test this out. We construct a question that would typically require an iterative sequence of retrieval steps to answer:


In [24]:
query = (
    "What is the standard method for Task Decomposition?\n\n"
    "Once you get the answer, look up common extensions of that method."
)
result = agent.invoke({"messages": [HumanMessage(query)]})

In [25]:
for msg in result["messages"]:
    msg.pretty_print()

================================ Human Message =================================

What is the standard method for Task Decomposition?

Once you get the answer, look up common extensions of that method.
================================== Ai Message ==================================
Tool Calls:
  retrieve_context (call_dab7dc1511da437d8a012a58)
 Call ID: call_dab7dc1511da437d8a012a58
  Args:
    query: standard method for Task Decomposition
================================= Tool Message =================================
Name: retrieve_context

Source: {'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'start_index': 2578}
Content: Task decomposition can be done (1) by LLM with simple prompting like "Steps for XYZ.\n1.", "What are the subgoals for achieving XYZ?", (2) by using task-specific instructions; e.g. "Write a story outline." for writing a novel, or (3) with human inputs.
Another quite distinct approach, LLM+P (Liu et al. 2023), involves relying on an external cla

Note that the agent:

1. Generates a query to search for a standard method for task decomposition;
2. Receiving the answer, generates a second query to search for common extensions of it;
3. Having received all necessary context, answers the question.

We can see the full sequence of steps, along with latency and other metadata, in the [LangSmith trace](https://smith.langchain.com/public/7b42d478-33d2-4631-90a4-7cb731681e88/r).

::: {.callout-tip}

You can add a deeper level of control and customization using the [LangGraph](https://docs.langchain.com/oss/python/langgraph/overview) framework directly— for example, you can add steps to grade document relevance and rewrite search queries. Check out LangGraph's [Agentic RAG tutorial](https://docs.langchain.com/oss/python/langgraph/agentic-rag) for more advanced formulations.

:::

### Agent vs Chain

In the above agentic RAG formulation **we allow the LLM to use its discretion** in generating a [tool call](/oss/python/langchain/models#tool-calling) to help answer user queries. This is a good general-purpose solution, but comes with some trade-offs:

#### ✅ Benefits

1. **Search only when needed** – The LLM can handle greetings, follow-ups, and simple queries without triggering unnecessary searches.
2. **Contextual search queries** – By treating search as a tool with a query input, the LLM crafts its own queries that incorporate conversational context.
3. **Multiple searches allowed** – The LLM can execute several searches in support of a single user query.

#### ⚠️ Drawbacks

1. **Two inference calls** – When a search is performed, it requires one call to generate the query and another to produce the final response.
2. **Reduced control** – The LLM may skip searches when they are actually needed, or issue extra searches when unnecessary.

## Key Takeaways

1. **RAG** (Retrieval Augmented Generation) combines retrieval from your data with an LLM to answer questions over specific source information.
2. **Indexing vs. retrieval** – Indexing (ingesting and embedding data into a vector store) is usually a separate, offline step; retrieval and generation happen at query time.
3. **RAG chain** – Single LLM call per query: retrieve docs → pass to model → answer. Fast and simple for straightforward questions.
4. **RAG agent** – Uses a search tool so the LLM decides when and how to search. Better for multi-step or nuanced queries (e.g., multiple searches, contextual query rewriting).
5. **Agent trade-offs** – Agents can skip unnecessary searches and issue multiple or contextual queries, but add an extra inference call when they do search and give you less direct control than a fixed chain.